# Controllability Summary Analysis

This notebook compares controllability across all studies at once.

It is meant to answer:
- how each model performs on Study A, B, and C controllability
- how the benchmark-level controllability score compares across models


In [ ]:
import sys
from pathlib import Path

for candidate_root in [Path.cwd(), Path.cwd().parent]:
    src_dir = candidate_root / "src"
    if src_dir.exists() and str(src_dir.resolve()) not in sys.path:
        sys.path.insert(0, str(src_dir.resolve()))

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

RESULTS_DIR = next(
    (p for p in [Path("results"), Path("../results"), Path("../../results")] if p.exists()),
    Path("results"),
)
print(f"Using RESULTS_DIR: {RESULTS_DIR.resolve()}")


In [ ]:
def load_controllability_summary_rows():
    rows = []
    if not RESULTS_DIR.exists():
        return pd.DataFrame()
    for model_dir in sorted(path for path in RESULTS_DIR.iterdir() if path.is_dir()):
        summary_path = model_dir / "controllability_summary.json"
        if not summary_path.exists():
            continue
        payload = json.loads(summary_path.read_text(encoding="utf-8"))
        studies = payload.get("studies", {})
        rows.append(
            {
                "model": payload.get("model", model_dir.name),
                "benchmark_control_score": (payload.get("benchmark_control") or {}).get("score"),
                "study_a_control": ((studies.get("A") or {}).get("aggregate") or {}).get("score"),
                "study_b_control": ((studies.get("B") or {}).get("aggregate") or {}).get("score"),
                "study_c_control": ((studies.get("C") or {}).get("aggregate") or {}).get("score"),
            }
        )
    return pd.DataFrame(rows)

summary_df = load_controllability_summary_rows()
if summary_df.empty:
    print("No controllability summary files found yet.")
else:
    display(summary_df.sort_values("benchmark_control_score", ascending=False).reset_index(drop=True))


## Benchmark Control


In [ ]:
if summary_df.empty:
    print("Skipping benchmark control plot - no summary rows found.")
else:
    plot_df = summary_df.sort_values("benchmark_control_score", ascending=False)
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.bar(plot_df["model"], plot_df["benchmark_control_score"], color="#8172B2", alpha=0.85)
    ax.set_title("Benchmark-Level Controllability", fontsize=14, fontweight="bold")
    ax.set_xlabel("Model")
    ax.set_ylabel("Benchmark control score")
    plt.xticks(rotation=45, ha="right")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


## Study Aggregate Comparison


In [ ]:
if summary_df.empty:
    print("Skipping study aggregate comparison - no summary rows found.")
else:
    long_df = summary_df.melt(
        id_vars=["model"],
        value_vars=["study_a_control", "study_b_control", "study_c_control"],
        var_name="study",
        value_name="aggregate_score",
    ).dropna(subset=["aggregate_score"])

    fig, ax = plt.subplots(figsize=(16, 7))
    sns.barplot(data=long_df, x="study", y="aggregate_score", hue="model", ax=ax)
    ax.set_title("Study-Level Controllability Aggregates", fontsize=14, fontweight="bold")
    ax.set_xlabel("Study")
    ax.set_ylabel("Aggregate control score")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
